In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install ipywidgets

In [ ]:
import os
import pandas as pd
import torch

from transformers import T5Tokenizer, T5ForConditionalGeneration

CSV_PATH = "/content/drive/MyDrive/spells_master.csv"
MODEL_NAME = "google/flan-t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [2]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)

(1333, 20)


,name,desc,higher_levels,level,school,classes,subclasses,cast_time,range,duration,verbal,somatic,material,material_desc,concentration,ritual,damage_type,dc_type,attack_type,source
0,Prismatic Wall,"A shimmering, multicolored plane of light form...",NaN,9,Abjuration,['Wizard'],[],1 action,60 feet,10 minutes,True,True,False,NaN,False,False,NaN,NaN,NaN,wotc-srd
1,Symbol,"When you cast this spell, you inscribe a harmf...",NaN,7,Abjuration,"['Bard', 'Cleric', 'Wizard']",[],1 minute,Touch,Until dispelled or triggered,True,True,True,"Mercury, phosphorus, and powdered diamond and ...",False,False,NaN,NaN,NaN,wotc-srd
2,Teleport,This spell instantly transports you and up to ...,NaN,7,Conjuration,"['Bard', 'Sorcerer', 'Wizard']",[],1 action,10 feet,Instantaneous,True,False,False,NaN,False,False,NaN,NaN,NaN,wotc-srd


In [3]:
torch.cuda.empty_cache()
import gc
gc.collect()

222

In [4]:
import torch
torch.cuda.empty_cache()

print(round(torch.cuda.memory_allocated() / 1e9, 2), 'GB used')
print(round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB total')

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
# bfloat16: native on L4/A100, same exponent range as float32 → no NaN risk
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
model = model.to(device)

print('model loaded')
print(round(torch.cuda.memory_allocated() / 1e9, 2), 'GB used after load')


0.0 GB used
23.66 GB total


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model loaded
1.91 GB used after load


In [5]:
prompt = "describe spell: Fireball"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40
    )

result = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(result)

fireball


In [6]:
df = df.dropna(subset=["name", "desc"])

df["name"] = df["name"].astype(str)
df["desc"] = df["desc"].astype(str)

df = df[df["desc"].str.len() > 20]

print(len(df))

1333


In [7]:
class SpellDataset(torch.utils.data.Dataset):
    """Tokenizes all examples once at construction time.
    __getitem__ does zero CPU work — tensors are ready to go straight to GPU."""

    def __init__(self, data):
        inputs = tokenizer(
            [item["input"]  for item in data],
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        targets = tokenizer(
            [item["target"] for item in data],
            max_length=256,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        label_ids = targets["input_ids"].clone()
        label_ids[label_ids == tokenizer.pad_token_id] = -100  # ignore padding in loss

        self.input_ids      = inputs["input_ids"]
        self.attention_mask = inputs["attention_mask"]
        self.labels         = label_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }


In [8]:
def build_spell_prompt(row):

    fields = []

    fields.append(f"level: {row['level']}")

    if "school" in row and pd.notna(row["school"]):
        fields.append(f"school: {row['school']}")

    if "cast_time" in row and pd.notna(row["cast_time"]):
        fields.append(f"cast_time: {row['cast_time']}")

    if "range" in row and pd.notna(row["range"]):
        fields.append(f"range: {row['range']}")

    if "duration" in row and pd.notna(row["duration"]):
        fields.append(f"duration: {row['duration']}")

    if "damage_type" in row and pd.notna(row["damage_type"]):
        fields.append(f"damage: {row['damage_type']}")

    return " | ".join(fields)

In [ ]:
multi_task_examples = []

for _, row in df.iterrows():

    name = str(row["name"])
    desc = str(row["desc"])

    attrs = build_spell_prompt(row)

    multi_task_examples.append({
        "input": f"describe spell: {name}",
        "target": desc,
        "task": "name_to_desc"
    })

    multi_task_examples.append({
        "input": f"generate name: {desc}",
        "target": name,
        "task": "desc_to_name"
    })

    multi_task_examples.append({
        "input": f"generate description: {attrs}",
        "target": desc,
        "task": "attr_to_desc"
    })

    if pd.notna(row.get("higher_levels")) and str(row["higher_levels"]).strip():
        multi_task_examples.append({
            "input":  f"upcast: {name} | {attrs}",
            "target": str(row["higher_levels"]),
            "task":   "upcast"
        })

    if pd.notna(row.get("school")) and str(row["school"]).strip():
        multi_task_examples.append({
            "input":  f"school of magic: {desc}",
            "target": str(row["school"]),
            "task":   "pred_school"
        })

print(len(multi_task_examples))

pd.DataFrame(multi_task_examples).head()

5849


,input,target,task
0,describe spell: Prismatic Wall,"A shimmering, multicolored plane of light form...",name_to_desc
1,"generate name: A shimmering, multicolored plan...",Prismatic Wall,desc_to_name
2,generate description: level: 9 | school: Abjur...,"A shimmering, multicolored plane of light form...",attr_to_desc
3,"school of magic: A shimmering, multicolored pl...",Abjuration,pred_school
4,describe spell: Symbol,"When you cast this spell, you inscribe a harmf...",name_to_desc


In [ ]:
import gc

def find_max_batch_size(model, tokenizer, device, start=128, min_bs=4):
    """
    Binary search for the largest batch size that fits in GPU memory.
    Runs a full forward + backward pass at each candidate size.
    Returns the largest size that doesn't OOM.
    """
    dummy_in  = "describe spell: Fireball"
    dummy_out = "A bright streak of light flashes to a point and explodes in a roar of flame."

    low, high, best = min_bs, start, min_bs

    while low <= high:
        bs = (low + high) // 2
        torch.cuda.empty_cache()
        gc.collect()

        try:
            enc = tokenizer(
                [dummy_in] * bs,
                max_length=128, truncation=True,
                padding="max_length", return_tensors="pt",
            ).to(device)

            tgt = tokenizer(
                [dummy_out] * bs,
                max_length=256, truncation=True,
                padding="max_length", return_tensors="pt",
            )
            labels = tgt["input_ids"].clone().to(device)
            labels[labels == tokenizer.pad_token_id] = -100

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                loss = model(**enc, labels=labels).loss
            loss.backward()
            model.zero_grad(set_to_none=True)

            best = bs
            low  = bs + 1
            print(f"  bs={bs:4d}  ✓")

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            high = bs - 1
            print(f"  bs={bs:4d}  ✗ OOM")

    torch.cuda.empty_cache()
    gc.collect()
    return best


print("probing max batch size (forward + backward)…")
_max_bs = find_max_batch_size(model, tokenizer, device, start=128, min_bs=4)

# 80 % safety margin for train (has gradients + optimizer state)
# full size for val  (inference only — no gradient bookkeeping)
TRAIN_BATCH_SIZE = max(4, int(_max_bs * 0.8))
VAL_BATCH_SIZE   = _max_bs

print(f"\nmax that fits : {_max_bs}")
print(f"train batch   : {TRAIN_BATCH_SIZE}  (80 % of max)")
print(f"val   batch   : {VAL_BATCH_SIZE}")

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

multi_train, multi_val = train_test_split(
    multi_task_examples,
    test_size=0.1,
    random_state=42
)

# Pre-tokenizes everything once here — __getitem__ is now instant
train_dataset = SpellDataset(multi_train)
val_dataset   = SpellDataset(multi_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
)

print(f"train batch size : {TRAIN_BATCH_SIZE}")
print(f"val   batch size : {VAL_BATCH_SIZE}")
print(f"train batches    : {len(train_loader)}")
print(f"val   batches    : {len(val_loader)}")

In [ ]:
import shutil

# Clear old T5-base weights — they are incompatible with flan-t5-large
# Run this once before the first training run, then comment it out
shutil.rmtree("/content/drive/MyDrive/GenAiText/t5_spell_model", ignore_errors=True)
os.makedirs("/content/drive/MyDrive/GenAiText/t5_spell_model", exist_ok=True)
print("cleared — save dir is fresh")

cleared — save dir is fresh


In [ ]:
import math
import matplotlib.pyplot as plt
from transformers import get_cosine_schedule_with_warmup

EPOCHS   = 25
LR       = 5e-5
SAVE_DIR = "/content/drive/MyDrive/GenAiText/t5_spell_model"
os.makedirs(SAVE_DIR, exist_ok=True)
checkpoint_path = os.path.join(SAVE_DIR, "training_state.pt")

# bfloat16: native on L4, same exponent range as float32 → no NaN, no GradScaler needed
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).to(device)

# torch.compile traces the graph once → ~15-30% faster per step after the first epoch
# try:
#     model = torch.compile(model)
#     print("torch.compile: enabled (first epoch slower for tracing)")
# except Exception as e:
#     print(f"torch.compile: skipped ({e})")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)

start_epoch   = 0
best_val_loss = float("inf")
train_history      = []
val_history        = []
perplexity_history = []

print("ready — flan-t5-large, bfloat16, cosine scheduler")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


ready — flan-t5-large, bfloat16, cosine scheduler


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

def unwrap(m):
    return m._orig_mod if hasattr(m, "_orig_mod") else m


for epoch in range(start_epoch, EPOCHS):

    # ── Training ──────────────────────────────────────────────
    model.train()
    total_train_loss = 0

    for batch in train_loader:

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
            loss.backward()  # inside autocast

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_history.append(avg_train_loss)

    # ── Validation ────────────────────────────────────────────
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_loader:

            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_history.append(avg_val_loss)

    perplexity = math.exp(avg_val_loss)
    perplexity_history.append(perplexity)

    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"  Train Loss : {avg_train_loss:.4f}")
    print(f"  Val Loss   : {avg_val_loss:.4f}")
    print(f"  Perplexity : {perplexity:.2f}")

    # ── Checkpoint ────────────────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        raw = unwrap(model)
        raw.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)

        torch.save(
            {
                "epoch"             : epoch,
                "model_state"       : raw.state_dict(),
                "optimizer_state"   : optimizer.state_dict(),
                "scheduler_state"   : scheduler.state_dict(),
                "best_val_loss"     : best_val_loss,
                "train_history"     : train_history,
                "val_history"       : val_history,
                "perplexity_history": perplexity_history,
            },
            checkpoint_path,
        )
        print("  ✓ saved best model")

    print("-" * 45)
    torch.cuda.empty_cache()  # free fragments between epochs

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(perplexity_history)

plt.xlabel("Epoch")
plt.ylabel("Perplexity")

plt.title("Validation Perplexity")

plt.show()

In [ ]:
def generate(prompt, max_new_tokens=120):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

In [ ]:
probe_prompts = [

    "describe spell: Fireball",

    "describe spell: Misty Step",

    (
        "generate description: "
        "level: 3 | school: Necromancy | "
        "damage: Poison | duration: 1 minute"
    ),

    (
        "generate description: "
        "level: 7 | school: Conjuration | "
        "range: 500 feet | duration: Instantaneous"
    ),

    (
        "generate description: "
        "level: 2 | school: Illusion | "
        "duration: 10 minutes"
    ),

    (
        "generate description: "
        "level: 0 | school: Evocation | "
        "damage: Lightning | range: 60 feet"
    ),

    (
        "generate description: "
        "level: 6 | school: Transmutation | "
        "duration: 24 hours"
    ),

    (
        "generate name: "
        "A wave of freezing wind blasts outward from you, "
        "dealing cold damage to creatures in a cone."
    ),

    (
        "generate name: "
        "You summon spectral chains that restrain enemies "
        "and drain their life force."
    ),

    (
        "describe spell: Ashen Nova | "
        "school: Evocation | level: 5"
    ),

]

In [ ]:
for i, prompt in enumerate(probe_prompts):


    print(f"Prompt {i + 1}")

    print(prompt)

    print("Output:\n")

    result = generate(
        prompt,
        max_new_tokens=200
    )

    print(result)

    print("\n")